# gammaConversionSi — training the conversion flow

Trains `ConversionFlow` from `conversion_flow.py` on the conversion ntuple. Unlike
`train_dnn.ipynb`, this model **samples** the final state rather than predicting it:

    input    eGamma
    sampled  isTriplet, eRecoil, eLead, thetaLead
    derived  eSub = eGamma - 2*me - eLead - eRecoil

`pathInBlock` is not an input. Given that a conversion happened, the kinematics depend on the
photon energy and the material, not on where in the block it occurred — the vertex is sampled
independently, from the attenuation length, and in fast simulation Geant4 supplies it itself.

**One head per quantity, chained.** Each head has its own network reading a shared trunk over
`eGamma`, and each also sees the quantity sampled before it, so the product of the four heads is
the exact joint density by the chain rule. The `isTriplet` head goes first because `eRecoil` is
bimodal — nuclear recoil of tens of eV for ~93 % of conversions, an atomic electron carrying up to
two thirds of the photon energy for the rest.

**The loss is formed per head**: each head's negative log-likelihood is averaged over the batch on
its own and the four means are then summed, so one head cannot dominate the total, and a head that
stops learning shows up as its own flat curve.

The flow works in scaled coordinates (energy *fractions* and `theta·E/me`), which is what makes
energy conservation and `eSub >= 0` structural rather than learned. See the module docstring.

Nothing is drawn inline — every figure is saved and closed.

In [ ]:
%load_ext autoreload
%autoreload 2

import time
from pathlib import Path

import matplotlib.pyplot as plt
import mplhep as hep
import numpy as np
import torch
import uproot
from torch.utils.data import DataLoader, TensorDataset

from conversion_data import ELECTRON_MASS, NTUPLE_BRANCHES, build_dataset
from conversion_flow import LOSS_TERMS, ConversionFlow

hep.style.use(hep.style.ATLAS)

# TeX Gyre Heros has no sized-radical glyph, so the \sqrt in the ATLAS `com`
# label warns and falls back to '?'. stixsans supplies it; fontset stays
# "custom", so all other math is still TeX Gyre Heros.
plt.rcParams["mathtext.fallback"] = "stixsans"

# Geant4 keeps the house black outline; the flow gets a second hue *and* a
# dashed line, so the two are never told apart by colour alone.
TRUTH_STYLE = dict(color="black", linestyle="-")
FLOW_STYLE = dict(color="#eb6834", linestyle="--")

In [ ]:
def find_repo_root(start=None):
    """Walk up from `start` until a directory containing .git is found."""
    path = (start or Path.cwd()).resolve()
    for candidate in (path, *path.parents):
        if (candidate / ".git").exists():
            return candidate
    raise RuntimeError(f"no .git found above {path}")


REPO = find_repo_root()

NTUPLE = REPO / "build/gammaConversionSi/ntuples/Si_2MeV-100GeV_10000000.root"
PLOTS = REPO / "studies/gammaConversionSi/training/plots"
MODELS = REPO / "studies/gammaConversionSi/training/models"

# Feature plots and closure plots each go in their own subdirectory of plots/
INPUT_PLOT_SUBDIR = "flow_inputs"
CLOSURE_PLOT_SUBDIR = "flow_closure"
INPUT_PLOTS = PLOTS / INPUT_PLOT_SUBDIR
CLOSURE_PLOTS = PLOTS / CLOSURE_PLOT_SUBDIR

COM = "2MeV - 100GeV"

# Training configuration
MAX_EVENTS = None  # a flow costs more per epoch than the MLP; None for all
VAL_FRACTION = 0.4
BATCH_SIZE = 1024
EPOCHS = 50
LEARNING_RATE = 1e-5
SEED = 0

for directory in (INPUT_PLOTS, CLOSURE_PLOTS, MODELS):
    directory.mkdir(parents=True, exist_ok=True)

if not NTUPLE.exists():
    raise FileNotFoundError(
        f"{NTUPLE} not found. Generate it from the repository root with:\n"
        f"    ./studies/gammaConversionSi/build.sh\n"
        f"    source install/geant4/bin/geant4.sh\n"
        f"    cd build/gammaConversionSi && ./gammaConversionSi config/default.cfg"
    )

DEVICE = (
    "mps" if torch.backends.mps.is_available()
    else "cuda" if torch.cuda.is_available()
    else "cpu"
)

torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)

print(f"repository   : {REPO}")
print(f"ntuple       : {NTUPLE.relative_to(REPO)}")
print(f"feature plots: {INPUT_PLOTS.relative_to(REPO)}")
print(f"closure plots: {CLOSURE_PLOTS.relative_to(REPO)}")
print(f"device       : {DEVICE}")

In [ ]:
tree = uproot.open(NTUPLE)["conversions"]
arrays = tree.arrays(NTUPLE_BRANCHES, entry_stop=MAX_EVENTS, library="np")

x_all, y_all, extra = build_dataset(arrays)
e_sub_true = extra["eSub"]
is_triplet_all = extra["isTriplet"]

print(f"events: {len(x_all):,}")
print(f"input  {x_all.shape}   targets {y_all.shape}")
print(f"triplet fraction: {is_triplet_all.mean():.4f}   (expected 1/(Z+1) = 0.0667 in silicon)")

In [ ]:
def plot_feature(values, xlabel, filename, logy=False, logx=True, bins=100):
    """Histogram one training variable and save it into INPUT_PLOTS.

    Same style as the exploration notebook: black outline, no fill, Poisson
    error bars, saved and closed rather than shown.
    """
    values = np.asarray(values, dtype=float)

    if logx:
        values = values[values > 0]
        edges = np.logspace(np.log10(values.min()), np.log10(values.max()), bins + 1)
    else:
        edges = np.linspace(values.min(), values.max(), bins + 1)

    counts, edges = np.histogram(values, bins=edges)

    fig, ax = plt.subplots()
    hep.histplot(counts, edges, ax=ax, yerr=True, histtype="step", **TRUTH_STYLE)

    ax.set_xlabel(xlabel)
    ax.set_ylabel("Conversions")
    if logx:
        ax.set_xscale("log")

    peak = counts.max() + np.sqrt(counts.max())
    if logy:
        ax.set_yscale("log")
        floor = max(counts[counts > 0].min(), 1) if (counts > 0).any() else 1
        ax.set_ylim(0.5 * floor, peak * 10)
    else:
        ax.set_ylim(0, peak * 1.35)

    hep.atlas.label(text="Internal", com=COM, ax=ax)

    path = INPUT_PLOTS / filename
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    return path


FEATURES = [
    # model input
    (x_all[:, 0], r"$E_\gamma$ [MeV]",             "input_eGamma.pdf",     dict()),
    # sampled quantities
    (y_all[:, 0], r"$E_\mathrm{lead}$ [MeV]",      "target_eLead.pdf",     dict()),
    (y_all[:, 1], r"$\theta_\mathrm{lead}$ [rad]", "target_thetaLead.pdf", dict(logy=True)),
    (y_all[:, 2], r"$E_\mathrm{recoil}$ [MeV]",    "target_eRecoil.pdf",   dict(logy=True)),
    # derived, for reference
    (e_sub_true,  r"$E_\mathrm{sub}$ [MeV]",       "derived_eSub.pdf",     dict(logy=True)),
]

for values, xlabel, filename, kwargs in FEATURES:
    path = plot_feature(values, xlabel, filename, **kwargs)
    print(f"saved {path.relative_to(REPO)}")

In [ ]:
# Shuffle once, then split. The normalisation is fitted on the training half
# only -- fitting on everything would leak the validation set's range.
order = rng.permutation(len(x_all))
n_val = int(len(order) * VAL_FRACTION)
val_idx, train_idx = order[:n_val], order[n_val:]

x_train = torch.from_numpy(x_all[train_idx])
y_train = torch.from_numpy(y_all[train_idx])
t_train = torch.from_numpy(is_triplet_all[train_idx])

x_val = torch.from_numpy(x_all[val_idx]).to(DEVICE)
y_val = torch.from_numpy(y_all[val_idx]).to(DEVICE)
t_val = torch.from_numpy(is_triplet_all[val_idx]).to(DEVICE)

model = ConversionFlow().to(DEVICE)
model.fit_normalisation(x_train.to(DEVICE), y_train.to(DEVICE), t_train.to(DEVICE))

print(f"train {len(x_train):,}   validation {len(x_val):,}")
print(f"parameters: {sum(p.numel() for p in model.parameters()):,}")
print()
print(f"eGamma  log10 min/max  {model.input_min.tolist()} {model.input_max.tolist()}")
# z_recoil is standardised per conversion mode: [nuclear, triplet]. The gap
# between the two means is why the mode is sampled first.
print(f"z_recoil mu   (nuclear, triplet) {model.recoil_mu.tolist()}")
print(f"z_recoil sigma(nuclear, triplet) {model.recoil_sigma.tolist()}")
print(f"z_lead   mu/sigma  {model.lead_mu.item():.3f} {model.lead_sigma.item():.3f}")
print(f"z_theta  mu/sigma  {model.theta_mu.item():.3f} {model.theta_sigma.item():.3f}")

In [ ]:
loader = DataLoader(
    TensorDataset(x_train, y_train, t_train),
    batch_size=BATCH_SIZE,
    shuffle=True,
)

optimiser = model.make_optimiser(LEARNING_RATE)


@torch.no_grad()
def evaluate(x, y, t, chunk=131072):
    """Per-head validation NLL, in chunks.

    The validation split can be ~2M rows; pushing it through in one batch would
    allocate gigabytes of hidden activations at once.
    """
    model.eval()
    totals = {name: 0.0 for name in LOSS_TERMS}
    count = 0
    for start in range(0, len(x), chunk):
        stop = start + chunk
        _, per_head = model.loss(x[start:stop], y[start:stop], t[start:stop])
        n = len(x[start:stop])
        for name, value in per_head.items():
            totals[name] += value.item() * n
        count += n
    return {name: total / count for name, total in totals.items()}


def format_terms(terms):
    return "  ".join(f"{name} {terms[name]:+7.3f}" for name in LOSS_TERMS)


history = {"train": [], "val": [], "train_terms": [], "val_terms": [], "seconds": []}

for epoch in range(1, EPOCHS + 1):
    started = time.perf_counter()
    model.train()
    running = {name: 0.0 for name in LOSS_TERMS}
    batches = 0

    for xb, yb, tb in loader:
        xb, yb, tb = xb.to(DEVICE), yb.to(DEVICE), tb.to(DEVICE)
        # Each head's NLL is averaged on its own inside model.loss(), and the
        # four means are summed -- so a head whose density happens to sit on a
        # different scale cannot swamp the gradient the others contribute.
        loss, per_head = model.loss(xb, yb, tb)

        optimiser.zero_grad()
        loss.backward()
        optimiser.step()

        for name, value in per_head.items():
            running[name] += value.item()
        batches += 1

    train_terms = {name: total / batches for name, total in running.items()}
    val_terms = evaluate(x_val, y_val, t_val)

    # Both loops call .item() every batch, which blocks until the device has
    # caught up, so this is wall-clock work done and not queue time -- no
    # explicit mps/cuda synchronize needed before reading the clock.
    elapsed = time.perf_counter() - started

    history["train"].append(sum(train_terms.values()))
    history["val"].append(sum(val_terms.values()))
    history["train_terms"].append(train_terms)
    history["val_terms"].append(val_terms)
    history["seconds"].append(elapsed)

    print(
        f"epoch {epoch:3d}   train {history['train'][-1]:+8.3f}   "
        f"val {history['val'][-1]:+8.3f}   [{format_terms(val_terms)}]   {elapsed:6.1f} s"
    )

total = sum(history["seconds"])
print(f"\n{EPOCHS} epochs in {total / 60:.1f} min, "
      f"{total / EPOCHS:.1f} s/epoch on average ({DEVICE})")
print("NLL is in the model's scaled coordinates -- not a likelihood in MeV/rad,")
print("and not comparable with the MSE the DNN reports.")

In [ ]:
epochs = np.arange(1, len(history["train"]) + 1)

# Total, then one panel per head. The per-head panels are the point of keeping
# the terms separate: a head that has stopped learning is invisible in the sum.
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs, history["train"], label="Train", **TRUTH_STYLE)
axes[0].plot(epochs, history["val"], label="Validation", **FLOW_STYLE)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Summed NLL (scaled coordinates)")
axes[0].legend()

for name in LOSS_TERMS:
    axes[1].plot(epochs, [terms[name] for terms in history["val_terms"]], label=name)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Validation NLL per head")
axes[1].legend()

hep.atlas.label(text="Internal", com=COM, ax=axes[0])
fig.savefig(PLOTS / "flow_training_loss.pdf", bbox_inches="tight")
plt.close(fig)

print(f"first epoch val {history['val'][0]:+.3f}  ->  last {history['val'][-1]:+.3f}")
print(f"saved {(PLOTS / 'flow_training_loss.pdf').relative_to(REPO)}")

## Closure

The test of a sampler is not a per-event error — there is no right answer per event — but whether
the *distributions* it generates match Geant4's. One sample is drawn per validation row, at that
row's true `eGamma`, and compared against the truth for the same rows.

In [ ]:
@torch.no_grad()
def sample(x, chunk=131072):
    """One sampled final state per row of `x`, in chunks."""
    model.eval()
    parts = [model.sample(x[start:start + chunk]) for start in range(0, len(x), chunk)]
    return {key: torch.cat([p[key] for p in parts]).cpu().numpy() for key in parts[0]}


torch.manual_seed(SEED)
flow = sample(x_val)

truth = {
    "eLead": y_val[:, 0].cpu().numpy(),
    "thetaLead": y_val[:, 1].cpu().numpy(),
    "eRecoil": y_val[:, 2].cpu().numpy(),
    "eSub": e_sub_true[val_idx],
    "isTriplet": t_val.cpu().numpy(),
}
e_gamma_val = x_val[:, 0].cpu().numpy()

# Both of these are structural, not learned -- from_learned() splits the
# available energy into fractions, so a violation means the inverse map is
# wrong, not that the model is undertrained.
residual = np.abs(
    e_gamma_val - 2 * ELECTRON_MASS
    - flow["eLead"] - flow["eRecoil"] - flow["eSub"]
).max()
print(f"energy conservation residual : {residual:.3e} MeV")
print(f"minimum sampled eSub         : {flow['eSub'].min():.3e} MeV  (must be >= 0)")
print(f"sampled eLead >= eSub always : {bool((flow['eLead'] >= flow['eSub']).all())}")
print()
print(f"triplet fraction  Geant4 {truth['isTriplet'].mean():.4f}   flow {flow['isTriplet'].mean():.4f}")

In [ ]:
def plot_closure(name, xlabel, filename, logx=True, logy=True, bins=80, rows=None, note=None):
    """Overlay the sampled distribution on Geant4's, with a ratio panel.

    Binning is taken from the truth, so the ratio panel is on the truth's
    support; the fraction of samples falling outside it is printed rather than
    silently piled into the edge bins.
    """
    true_values = np.asarray(truth[name], dtype=float)
    flow_values = np.asarray(flow[name], dtype=float)
    if rows is not None:
        true_values, flow_values = true_values[rows], flow_values[rows]

    if logx:
        positive = true_values[true_values > 0]
        edges = np.logspace(np.log10(positive.min()), np.log10(positive.max()), bins + 1)
    else:
        edges = np.linspace(true_values.min(), true_values.max(), bins + 1)

    outside = np.mean((flow_values < edges[0]) | (flow_values > edges[-1]))

    true_counts, _ = np.histogram(true_values, bins=edges)
    flow_counts, _ = np.histogram(flow_values, bins=edges)

    fig, (ax, rax) = plt.subplots(
        2, 1, sharex=True, figsize=(8, 8),
        gridspec_kw={"height_ratios": [3, 1], "hspace": 0.07},
    )

    hep.histplot(true_counts, edges, ax=ax, yerr=True, histtype="step",
                 label="Geant4", **TRUTH_STYLE)
    hep.histplot(flow_counts, edges, ax=ax, yerr=True, histtype="step",
                 label="Flow", **FLOW_STYLE)
    ax.set_ylabel("Conversions")
    ax.legend()
    if logy:
        ax.set_yscale("log")

    with np.errstate(divide="ignore", invalid="ignore"):
        ratio = np.where(true_counts > 0, flow_counts / true_counts, np.nan)
    hep.histplot(ratio, edges, ax=rax, histtype="step", **FLOW_STYLE)
    rax.axhline(1.0, color="black", linewidth=1)
    rax.set_ylim(0.0, 2.0)
    rax.set_ylabel("Flow / Geant4")
    rax.set_xlabel(xlabel)
    if logx:
        rax.set_xscale("log")

    hep.atlas.label(text="Internal", com=COM, ax=ax)
    if note is not None:
        ax.text(0.05, 0.75, note, transform=ax.transAxes)

    path = CLOSURE_PLOTS / filename
    fig.savefig(path, bbox_inches="tight")
    plt.close(fig)
    print(f"saved {path.relative_to(REPO)}   ({outside * 100:.2f} % of samples outside the range)")


CLOSURES = [
    ("eLead",     r"$E_\mathrm{lead}$ [MeV]",      "closure_eLead.pdf",     dict()),
    ("eSub",      r"$E_\mathrm{sub}$ [MeV]",       "closure_eSub.pdf",      dict()),
    ("thetaLead", r"$\theta_\mathrm{lead}$ [rad]", "closure_thetaLead.pdf", dict()),
    ("eRecoil",   r"$E_\mathrm{recoil}$ [MeV]",    "closure_eRecoil.pdf",   dict()),
]

for name, xlabel, filename, kwargs in CLOSURES:
    plot_closure(name, xlabel, filename, **kwargs)

In [ ]:
# The same comparison inside narrow eGamma slices. An inclusive marginal can
# look right while every individual energy is wrong, because the slices average
# over each other.
SLICES = [(10.0, "10 MeV"), (1.0e3, "1 GeV"), (5.0e4, "50 GeV")]
SLICE_HALF_WIDTH = 0.2  # +-20 % in eGamma

for centre, label in SLICES:
    rows = np.abs(e_gamma_val / centre - 1.0) < SLICE_HALF_WIDTH
    if rows.sum() < 1000:
        print(f"slice {label}: only {rows.sum()} rows, skipped")
        continue
    stem = label.replace(" ", "")
    for name, xlabel, _, kwargs in CLOSURES:
        plot_closure(
            name, xlabel, f"slice_{stem}_{name}.pdf",
            rows=rows, note=rf"$E_\gamma \approx$ {label}", **kwargs
        )

In [ ]:
# The correlation the chained heads exist to reproduce. Independent heads would
# get both marginals right and this plot wrong.
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)

x_edges = np.logspace(np.log10(max(truth["eLead"].min(), 1e-3)), np.log10(truth["eLead"].max()), 80)
y_edges = np.logspace(np.log10(max(truth["thetaLead"][truth["thetaLead"] > 0].min(), 1e-8)),
                      np.log10(truth["thetaLead"].max()), 80)

panels = [("Geant4", truth), ("Flow", flow)]
counts = [np.histogram2d(p["eLead"], p["thetaLead"], bins=[x_edges, y_edges])[0] for _, p in panels]
vmax = max(c.max() for c in counts)

for ax, (title, _), count in zip(axes, panels, counts):
    mesh = ax.pcolormesh(x_edges, y_edges, count.T, cmap="Blues", vmin=0, vmax=vmax)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"$E_\mathrm{lead}$ [MeV]")
    ax.set_title(title)
axes[0].set_ylabel(r"$\theta_\mathrm{lead}$ [rad]")
fig.colorbar(mesh, ax=axes, label="Conversions")

path = CLOSURE_PLOTS / "closure_eLead_vs_thetaLead.pdf"
fig.savefig(path, bbox_inches="tight")
plt.close(fig)
print(f"saved {path.relative_to(REPO)}")

In [ ]:
path = MODELS / "conversion_flow.pt"
torch.save(model.state_dict(), path)

# The state_dict carries the normalisation buffers, so this is all that is
# needed for inference -- no scaler to reload alongside it.
reloaded = ConversionFlow().to(DEVICE)
reloaded.load_state_dict(torch.load(path))
reloaded.eval()

# Sampling is stochastic, so the check is that the same seed gives the same
# draw, not that two calls agree.
torch.manual_seed(SEED)
with torch.no_grad():
    before = model.sample(x_val[:1024])
torch.manual_seed(SEED)
with torch.no_grad():
    after = reloaded.sample(x_val[:1024])

identical = all(torch.equal(before[key], after[key]) for key in before)
print(f"saved {path.relative_to(REPO)}")
print(f"reloaded model reproduces samples exactly: {identical}")